Library install

In [ ]:
!pip install transformers datasets torch scikit-learn pandas accelerate -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
for f in os.listdir('/content/drive/MyDrive/499A_FakeNews'):
    print(f)

final_train_17k.tsv
images.zip


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/499A_FakeNews/final_train_17k.tsv', sep='\t')
print(df.shape)
print(df.columns.tolist())
df.head()

(17045, 16)
['author', 'clean_title', 'created_utc', 'domain', 'hasImage', 'id', 'image_url', 'linked_submission_id', 'num_comments', 'score', 'subreddit', 'title', 'upvote_ratio', '2_way_label', '3_way_label', '6_way_label']


,author,clean_title,created_utc,domain,hasImage,id,image_url,linked_submission_id,num_comments,score,subreddit,title,upvote_ratio,2_way_label,3_way_label,6_way_label
0,Alexithymia,my walgreens offbrand mucinex was engraved wit...,1.551641e+09,i.imgur.com,True,awxhir,https://external-preview.redd.it/WylDbZrnbvZdB...,NaN,2.0,12,mildlyinteresting,My Walgreens offbrand Mucinex was engraved wit...,0.84,1,0,0
1,prometheus1123,hackers leak emails from uae ambassador to us,1.496511e+09,aljazeera.com,True,6f2cy5,https://external-preview.redd.it/6fNhdbc6K1vFA...,NaN,1.0,44,neutralnews,Hackers leak emails from UAE ambassador to US,0.92,1,0,0
2,NaN,puppy taking in the view,1.471341e+09,i.imgur.com,True,4xypkv,https://external-preview.redd.it/HLtVNhTR6wtYt...,NaN,26.0,250,photoshopbattles,PsBattle: Puppy taking in the view,0.95,1,0,0
3,CrimsonBlue90,bride and groom exchange vows after fatal shoo...,1.423681e+09,independent.ie,True,2vkbtj,https://external-preview.redd.it/FQ-J9OIPFRpqi...,NaN,7.0,6,nottheonion,Bride and groom exchange vows after fatal shoo...,0.64,1,0,0
4,happenpupe,major thermos,1.495660e+09,i.redd.it,True,6d50rl,https://preview.redd.it/l9gvkkf3jizy.jpg?width...,NaN,0.0,2,pareidolia,major thermos,0.67,0,2,2


In [ ]:
df = df[['clean_title', '2_way_label']].dropna()
df = df.rename(columns={'clean_title': 'text', '2_way_label': 'label'})
print(df['label'].value_counts())

label
0    10404
1     6641
Name: count, dtype: int64


Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print(len(train_df), len(test_df))

13636 3409


Making Tokenizer and Dataset

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/13636 [00:00<?, ? examples/s]

Map:   0%|          | 0/3409 [00:00<?, ? examples/s]

Model Load And Training Setup

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

training_args = TrainingArguments(
    output_dir="/content/results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="/content/logs",
    load_best_model_at_end=True,
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOAR

Training

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.567230,0.548406,0.758287,0.744575,0.632771,0.904367
2,0.438228,0.405821,0.818422,0.777418,0.743978,0.814006
3,0.342942,0.410759,0.830742,0.788257,0.768790,0.808735


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2559, training_loss=0.4460272205765707, metrics={'train_runtime': 1202.7857, 'train_samples_per_second': 34.011, 'train_steps_per_second': 2.128, 'total_flos': 2690836763166720.0, 'train_loss': 0.4460272205765707, 'epoch': 3.0})

In [ ]:
import os
base_path = '/content/drive/MyDrive/499A_FakeNews'
print(os.listdir(base_path))

['final_train_17k.tsv', 'images.zip']


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/499A_FakeNews/text_model', exist_ok=True)

model.save_pretrained('/content/drive/MyDrive/499A_FakeNews/text_model')
tokenizer.save_pretrained('/content/drive/MyDrive/499A_FakeNews/text_model')

print("Saved! Files:")
print(os.listdir('/content/drive/MyDrive/499A_FakeNews/text_model'))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved! Files:
['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']


Model workings

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)

text = "Scientists discover new planet capable of supporting life"

# ============================================
# STEP 1: Tokenization
# ============================================
tokens = tokenizer.tokenize(text)
print("STEP 1 - Tokens:")
print(tokens)
print()

# ============================================
# STEP 2: Token ==>(ID)
# ============================================
inputs = tokenizer(text, return_tensors="pt")
print("STEP 2 - Token IDs:")
print(inputs['input_ids'])
print()

# ============================================
# STEP 3: Embedding - 768 - dimension vector
# ============================================
with torch.no_grad():
    embedding_output = base_model.embeddings(inputs['input_ids'])
print("STEP 3 - Embedding Shape:")
print(embedding_output.shape)  # [1, num_tokens, 768]
print("First 5:", embedding_output[0][0][:5])
print()

# ============================================
# STEP 4: Model ==> Pass (12 Attention Layer)
# ============================================
with torch.no_grad():
    outputs = base_model(**inputs)
print("STEP 4 - last Layers Output Shape:")
print(outputs.last_hidden_state.shape)  # [1, num_tokens, 768]
print()

# ============================================
# STEP 5: Final Prediction by Classification Model
# ============================================
from transformers import AutoModelForSequenceClassification

clf_model = AutoModelForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/499A_FakeNews/text_model"
)
clf_tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/499A_FakeNews/text_model"
)

clf_inputs = clf_tokenizer(text, return_tensors="pt")
with torch.no_grad():
    logits = clf_model(**clf_inputs).logits

print("STEP 5 - Raw Output Scores (Logits):")
print(logits)

probabilities = torch.softmax(logits, dim=1)
print("STEP 5 - Probability (Fake vs Real):")
print(f"Fake probability: {probabilities[0][0].item():.4f}")
print(f"Real probability: {probabilities[0][1].item():.4f}")

prediction = torch.argmax(probabilities, dim=1).item()
print(f"\n Prediction: {'FAKE' if prediction == 0 else 'REAL'}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


STEP 1 - Tokens:
['▁S', 'cient', 'ists', '▁discover', '▁new', '▁planet', '▁capable', '▁of', '▁support', 'ing', '▁life']

STEP 2 - Token IDs:
tensor([[     0,    159,  45964,  64370, 103882,   3525,  23208,  87709,    111,
           8060,    214,   6897,      2]])

STEP 3 - Embedding Shape:
torch.Size([1, 13, 768])
First 5: tensor([-0.1478,  0.1493, -0.1428, -0.0122,  0.2529])

STEP 4 - last Layers Output Shape:
torch.Size([1, 13, 768])



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

STEP 5 - Raw Output Scores (Logits):
tensor([[ 0.5088, -0.7405]])
STEP 5 - Probability (Fake vs Real):
Fake probability: 0.7772
Real probability: 0.2228

 Prediction: FAKE


MODEL = xmlroberta

In [ ]:
print(model)

XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=768, out_features=2, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Li